## Machine Learning

In [ ]:
# Create a mean of the topspoil and subsoil values
soilpoints_avg = soilpoints.pivot_table(index=('Latitude', 'Longitude'), values=['Altitude', 'B11', 'B12', 'B2', 'B3', 'B4', 'B5', 'B6',
    'B7', 'B8', 'B8A', 'B9', 'VH', 'VV','predSOC'], aggfunc='mean').reset_index()
soilpoints_gdf = gpd.GeoDataFrame(soilpoints_avg, geometry=gpd.points_from_xy(soilpoints_avg.Longitude, soilpoints_avg.Latitude), crs='EPSG:4326')

In [ ]:
soilpoints_gdf.dropna(inplace=True)
predic = soilpoints_gdf['predSOC']
features = soilpoints_gdf.drop(columns=['predSOC', 'Latitude', 'Longitude', 'geometry'])

In [ ]:
# Train test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(features, predic, test_size=0.2, random_state=42)

In [ ]:
# Grid search of Random Forest, Gradient Boosting and SVM
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error as mse, mean_absolute_percentage_error as mape

In [ ]:
# Random Forest
rf = RandomForestRegressor()
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30]
}

rf_grid = GridSearchCV(rf, param_grid, cv=5, n_jobs=-1)
rf_grid.fit(X_train, y_train)
print(f'Random Forest Best Parameters: {rf_grid.best_params_}')
print(f'Random Forest Best Score: {rf_grid.best_score_}')

# Gradient Boosting
gb = GradientBoostingRegressor()
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30]
}

gb_grid = GridSearchCV(gb, param_grid, cv=5, n_jobs=-1)
gb_grid.fit(X_train, y_train)
print(f'Gradient Boosting Best Parameters: {gb_grid.best_params_}')
print(f'Gradient Boosting Best Score: {gb_grid.best_score_}')

# SVM
svm = SVR()
param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf']
}

svm_grid = GridSearchCV(svm, param_grid, cv=5, n_jobs=-1)
svm_grid.fit(X_train, y_train)
print(f'SVM Best Parameters: {svm_grid.best_params_}')
print(f'SVM Best Score: {svm_grid.best_score_}')

Random Forest Best Parameters: {'max_depth': 20, 'n_estimators': 300}
Random Forest Best Score: 0.4315354124254611
Gradient Boosting Best Parameters: {'max_depth': 10, 'n_estimators': 100}
Gradient Boosting Best Score: 0.31520435934358837
SVM Best Parameters: {'C': 1, 'kernel': 'linear'}
SVM Best Score: 0.05290174252058007


In [ ]:
# Random Forest
rf = RandomForestRegressor(n_estimators=300, max_depth=30)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_mse = mse(y_test, rf_pred)
rf_mape = mape(y_test, rf_pred)
print(f'Random Forest MSE: {rf_mse}') 
print(f'Random Forest MAPE: {rf_mape}')

Random Forest MSE: 5.0386354269554845
Random Forest MAPE: 0.14295177932597702
